# 04 - Genie validation

Manual verification of Genie answers. For each question, we run our own SQL
and check that Genie's numbers match.

*Some parts and solutions of this code was found through help from Claude*

In [0]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from pyspark.sql import functions as F

## Question 1: Monthly event count over the years

Genie's claim: 897 months, average 93.3 events per month, peak 920 events.
Genie's interpretation: distinct event_ids per (year, month).

In [0]:
result = spark.sql("""
    SELECT
        COUNT(*) AS num_months,
        ROUND(AVG(event_count), 2) AS avg_events_per_month,
        MAX(event_count) AS peak_events
    FROM (
        SELECT
            d.year,
            d.month,
            COUNT(DISTINCT r.event_id) AS event_count
        FROM marathos.gold.fct_results r
        JOIN marathos.gold.dim_date d ON r.start_date_id = d.date_id
        GROUP BY d.year, d.month
    )
""")

result.show()

**Match: 897 / 93.26 / 920 – matches Genie exactly.** ✓

## Question 2: Average finish time for 100km races by gender

Genie's claim:
- F: 16.9 hours
- M: 14.9 hours
- X: 16.0 hours

In [0]:
result = spark.sql("""
    SELECT
        a.athlete_gender,
        COUNT(*) AS num_results,
        ROUND(AVG(r.performance_seconds) / 3600, 2) AS avg_hours
    FROM marathos.gold.fct_results r
    JOIN marathos.gold.dim_athlete a ON r.athlete_id = a.athlete_id
    JOIN marathos.gold.dim_event e ON r.event_id = e.event_id
    WHERE e.event_distance_value = 100
      AND e.event_distance_unit = 'km'
      AND e.event_type = 'distance'
      AND r.performance_seconds IS NOT NULL
      AND a.athlete_gender IS NOT NULL
    GROUP BY a.athlete_gender
    ORDER BY a.athlete_gender
""")

result.show()

**Match: F=16.86, M=14.93, X=15.96 – matches Genie within rounding.** ✓

Note: X has only 2 results, so its average is not statistically meaningful.
Worth flagging to stakeholders when answering questions about minority categories.

## Question 3: Top 5 countries by longest 24-hour race distance

Genie's claim:
- Lithuania: 319.6 km
- Australia: 303.5 km
- Poland: 301.9 km
- Ukraine: 295.4 km
- Italy: 288.4 km

Genie ranks countries by MAX(performance_distance) per country.

In [0]:
result = spark.sql("""
    SELECT
        c.country_name,
        ROUND(MAX(r.performance_distance), 1) AS max_distance_km
    FROM marathos.gold.fct_results r
    JOIN marathos.gold.dim_event e ON r.event_id = e.event_id
    JOIN marathos.gold.dim_country c ON r.country_code = c.country_code
    WHERE e.event_distance_value = 24
      AND e.event_distance_unit = 'h'
      AND r.performance_distance IS NOT NULL
    GROUP BY c.country_name
    ORDER BY max_distance_km DESC
    LIMIT 5
""")

result.show(truncate=False)

**Match: All 5 countries and distances match Genie exactly.** ✓

Interesting finding: Lithuania tops the list - this is driven by Aleksandr Sorokin,
one of the all-time strongest 24-hour runners. Genie surfaces patterns that aren't
obvious from looking at participation numbers.